
# System predykcji zagrożenia finansowego<br>i ryzyka bankructwa przedsiębiorstw<br>z wykorzystaniem modeli uczenia maszynowego

### Autorzy: **Maria Wilgosz · Oliwia Marut · Martyna Czyż · Jakub Klimek**

# Rozdział 1: Wstęp

## 1.1 Cel projektu
bla bla czemu to robimy i wprowadzenie w temat o co chodzi

Przydatne zdania (?) Celem pracy jest zbadanie różnic między firmami oznaczonymi jako działające i upadłe oraz sprawdzenie, czy dostępne cechy finansowe pozwalają je rozróżnić za pomocą metod uczenia maszynowego.

## 1.2 Wykorzystane biblioteki

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

# Rozdział 2: Przedstawienie oraz przygotowanie danych

W projekcie wykorzystano zbiór American Bankruptcy Dataset, zawierający dane finansowe przedsiębiorstw z lat 1999–2018 oraz informacje o ich statusie. Zakres obejmuje opis kolumn, wczytanie pliku, sprawdzenie braków i duplikatów oraz uporządkowanie typów danych i kodowania statusu firmy.

## 2.1 Opis zbioru

Każdy wiersz opisuje jedną firmę w jednym roku obrotowym. Zbiór zawiera następujące kolumny:

- `company_name` — identyfikator firmy.
- `fyear` — rok obrotowy.
- `status_label` — status firmy: `alive` lub `failed`.
- `X1`–`X18` — wartości finansowe opisane poniżej.
- `Division`, `MajorGroup` — kody grup działalności. $\textbf{(POPRAW)}$

| Kod | Znaczenie | Kod | Znaczenie |
|---|---|---|---|
| X1 | Aktywa obrotowe — majątek wykorzystywany w krótkim okresie, np. zapasy, należności i gotówka. | X10 | Aktywa ogółem — łączna wartość majątku firmy. |
| X2 | Koszt własny sprzedaży — koszty sprzedanych produktów, towarów lub usług. | X11 | Dług długoterminowy — zadłużenie z terminem spłaty dłuższym niż rok. |
| X3 | Amortyzacja — rozłożony w czasie koszt zużycia majątku firmy. | X12 | EBIT — wynik finansowy przed uwzględnieniem odsetek i podatków. |
| X4 | EBITDA — wynik finansowy przed uwzględnieniem odsetek, podatków i amortyzacji. | X13 | Zysk brutto ze sprzedaży — przychody ze sprzedaży pomniejszone o koszt własny sprzedaży. |
| X5 | Zapasy — materiały, towary i produkty przeznaczone do wykorzystania lub sprzedaży. | X14 | Zobowiązania krótkoterminowe — zobowiązania do uregulowania zwykle w ciągu roku. |
| X6 | Wynik netto — ostateczny zysk lub strata po uwzględnieniu kosztów i podatków. | X15 | Zyski zatrzymane — skumulowane zyski pozostawione w firmie, pomniejszone o ewentualne straty. |
| X7 | Należności ogółem — kwoty, które inne podmioty powinny zapłacić firmie. | X16 | Przychody ogółem — łączne przychody uzyskane przez firmę. |
| X8 | Wartość rynkowa — wycena firmy na rynku, zwykle na podstawie wartości jej akcji. | X17 | Zobowiązania ogółem — suma zobowiązań krótko- i długoterminowych. |
| X9 | Sprzedaż netto — przychody ze sprzedaży po uwzględnieniu rabatów i zwrotów. | X18 | Koszty operacyjne — koszty związane z prowadzeniem działalności firmy. |


## 2.2. Kontrola jakości danych

W ramach wstępnej kontroli jakości danych sprawdzono występowanie brakujących wartości, zduplikowanych obserwacji oraz wartości nieskończonych w kolumnach liczbowych. Dodatkowo zweryfikowano, czy w zbiorze nie występują powtarzające się obserwacje dla tej samej pary przedsiębiorstwo–rok (`company_name`, `fyear`). Kontrola ta ma na celu wykrycie potencjalnych problemów mogących zaburzyć późniejszą analizę statystyczną oraz proces uczenia modeli.

In [8]:
df = pd.read_csv("american_bankruptcy_dataset.csv")

print("Łączna liczba braków:", df.isna().sum().sum())

print("Liczba duplikatów wierszy:", df.duplicated().sum())

print(
    "Duplikaty przedsiębiorstwo-rok:",
    df.duplicated(subset=["company_name", "fyear"]).sum()
)

kolumny_liczbowe = df.select_dtypes(include="number")

print(
    "Liczba wartości nieskończonych:",
    np.isinf(kolumny_liczbowe).sum().sum()
)

display(df.isna().sum().to_frame("Liczba braków"))

Łączna liczba braków: 0
Liczba duplikatów wierszy: 0
Duplikaty przedsiębiorstwo-rok: 0
Liczba wartości nieskończonych: 0


,Liczba braków
company_name,0
fyear,0
status_label,0
X1,0
X2,0
X3,0
X4,0
X5,0
X6,0
X7,0


W przeprowadzonej kontroli nie wykryto brakujących wartości w żadnej z kolumn zbioru. Nie stwierdzono również duplikatów obserwacji ani wartości nieskończonych, co pozwala uznać dane za kompletne i spójne na tym etapie analizy.

# Rozdział 3: Analiza danych finansowych

W rozdziale tym przeprowadzona zostanie analiza matematyczna i statystyczna parametrów finansowych zawartych w zbiorze danych. Celem analizy będzie określenie, w jaki sposób poszczególne zmienne zachowują się w badanej próbie, jak duża jest ich zmienność oraz jakie zależności występują pomiędzy nimi.

W pierwszej kolejności zostaną wyznaczone podstawowe statystyki opisowe dla parametrów X1–X18, takie jak średnia, mediana, odchylenie standardowe, wartości minimalne i maksymalne oraz kwartyle. Pozwoli to ocenić rozkład poszczególnych zmiennych, ich skalę oraz występowanie wartości skrajnych.

Następnie przeanalizowane zostaną zależności pomiędzy parametrami finansowymi, w tym ich wzajemne korelacje oraz relacje z klasą status_label. Szczególna uwaga zostanie poświęcona porównaniu firm oznaczonych jako alive oraz failed, aby sprawdzić, czy wartości wybranych parametrów różnią się pomiędzy tymi grupami.

## 3.1 Statystyki opisowe parametrów finansowych

W celu wstępnego scharakteryzowania zmiennych finansowych przeprowadzona zostanie analiza podstawowych statystyk opisowych dla parametrów X1–X18. Jej celem jest określenie skali poszczególnych zmiennych, ich zróżnicowania oraz sposobu rozłożenia wartości w całym zbiorze danych.

Dla każdego parametru zostaną zestawione takie miary jak średnia, mediana, odchylenie standardowe, kwartyle oraz wartości minimalne i maksymalne. Pozwoli to wskazać zmienne o dużej zmienności, zauważyć ewentualne wartości skrajne oraz ocenić, w których przypadkach rozkład danych może być silnie asymetryczny.

Zestawienie wszystkich parametrów w jednej tabeli umożliwi również ich porównanie pod względem skali i rozrzutu wartości. Wyniki tej analizy będą stanowiły punkt wyjścia do dalszego badania różnic pomiędzy firmami oznaczonymi jako alive i failed oraz zależności występujących pomiędzy poszczególnymi cechami finansowymi.

In [9]:
# wybór parametrów finansowych
parametry = [f"X{i}" for i in range(1, 19)]

# opis parametrów
nazwy_parametrow = {
    "X1": "X1 (Aktywa obrotowe)",
    "X2": "X2 (Koszt własny sprzedaży)",
    "X3": "X3 (Amortyzacja)",
    "X4": "X4 (EBITDA)",
    "X5": "X5 (Zapasy)",
    "X6": "X6 (Wynik netto)",
    "X7": "X7 (Należności ogółem)",
    "X8": "X8 (Wartość rynkowa)",
    "X9": "X9 (Sprzedaż netto)",
    "X10": "X10 (Aktywa ogółem)",
    "X11": "X11 (Dług długoterminowy)",
    "X12": "X12 (EBIT)",
    "X13": "X13 (Zysk brutto ze sprzedaży)",
    "X14": "X14 (Zobowiązania krótkoterminowe)",
    "X15": "X15 (Zyski zatrzymane)",
    "X16": "X16 (Przychody ogółem)",
    "X17": "X17 (Zobowiązania ogółem)",
    "X18": "X18 (Koszty operacyjne)"
}

# tabela statystyk opisowych
tabela_statystyk = pd.DataFrame({
    "Średnia": df[parametry].mean(),
    "Mediana": df[parametry].median(),
    "Odch.std": df[parametry].std(),
    "Q1": df[parametry].quantile(0.25),
    "Q3": df[parametry].quantile(0.75),
    "Min": df[parametry].min(),
    "Max": df[parametry].max()
})

# zmiana nazw parametrów w indeksie
tabela_statystyk.index = tabela_statystyk.index.map(nazwy_parametrow)
tabela_statystyk.index.name = "Parametr"

tabela_statystyk.round(2)

,Średnia,Mediana,Odch.std,Q1,Q3,Min,Max
Parametr,,,,,,,
X1 (Aktywa obrotowe),213314.08,22016.50,701629.10,1272.00,153994.00,-2498.00,31322583.0
X2 (Koszt własny sprzedaży),599470.89,43266.50,2065451.51,3282.52,329362.00,0.01,67234944.0
X3 (Amortyzacja),379774.46,21044.50,2021659.41,1439.10,180821.50,-366645.00,163245615.0
X4 (EBITDA),150843.34,500.00,711505.23,0.00,16010.50,-23.00,20687082.0
X5 (Zapasy),24715.65,1216.00,97781.69,74.40,12344.75,0.00,4440438.0
X6 (Wynik netto),44261.77,303.00,224001.73,-767.75,19097.75,-5043795.00,9783515.0
X7 (Należności ogółem),69421.15,1121.90,284799.37,-9.00,33256.75,-4512135.00,12566722.0
X8 (Wartość rynkowa),172375.67,11459.50,600752.29,355.43,109900.00,-4367571.00,27864399.0
X9 (Sprzedaż netto),51021.66,675.00,220407.62,0.00,16771.75,0.00,11918508.0


### Wnioski ze statystyk opisowych

Zestawione statystyki opisowe wskazują na bardzo duże zróżnicowanie wartości poszczególnych parametrów finansowych. W przypadku wielu zmiennych średnia jest wyraźnie większa od mediany, co sugeruje występowanie rozkładów silnie asymetrycznych oraz pojedynczych obserwacji o bardzo dużych wartościach. Jest to szczególnie widoczne m.in. dla EBITDA (`X4`), zysków zatrzymanych (`X15`), aktywów ogółem (`X10`) czy przychodów ogółem (`X16`). Oznacza to, że w zbiorze znajdują się przedsiębiorstwa bardzo różniące się skalą działalności, dlatego sama średnia nie zawsze będzie dobrą miarą wartości typowej.

Duże wartości odchylenia standardowego również potwierdzają znaczny rozrzut danych. Szczególnym przykładem jest `X15` (zyski zatrzymane), dla którego mediana wynosi około 373, podczas gdy wartość maksymalna przekracza 467 mln. Podobne, choć mniej skrajne, różnice występują także w innych parametrach. W dalszej analizie zasadne będzie zatem dokładniejsze sprawdzenie rozkładów oraz obserwacji odstających.

Warto zwrócić uwagę na parametry, które przyjmują zarówno wartości dodatnie, jak i ujemne. `X6` (wynik netto) ma pierwszy kwartyl poniżej zera, co oznacza, że znaczna część obserwacji dotyczy przedsiębiorstw wykazujących stratę. Jeszcze ciekawszy jest `X12` (EBIT), którego mediana jest lekko ujemna, a wartości rozciągają się od dużych strat do bardzo wysokich wyników dodatnich. Parametry te mogą być szczególnie interesujące przy późniejszym porównaniu przedsiębiorstw `alive` oraz `failed`.

W danych widoczne są również wartości wymagające dodatkowej weryfikacji. Niektóre zmienne posiadają nietypowe wartości minimalne, w tym wartości ujemne w parametrach, dla których z ekonomicznego punktu widzenia mogą one wymagać dodatkowego wyjaśnienia. Przed wykorzystaniem danych w modelach uczenia maszynowego należy więc sprawdzić konkretne rekordy i ustalić, czy wynikają one ze sposobu raportowania danych, zastosowanej konwencji księgowej czy też stanowią obserwacje nietypowe.

Szczególną uwagę zwracają parametry `X14` oraz `X17`. Ich średnia, mediana, odchylenie standardowe, kwartyle, minimum oraz maksimum są identyczne. Samo zestawienie statystyk nie pozwala stwierdzić, że kolumny są identyczne, jednak podobieństwo to jest na tyle duże, że należy sprawdzić zależność pomiędzy nimi bezpośrednio w danych. Jeżeli wartości obu parametrów okażą się takie same dla wszystkich obserwacji, będzie to istotne przy późniejszej analizie korelacji i doborze cech do modelu.

Na tym etapie statystyki opisowe pozwalają przede wszystkim scharakteryzować strukturę zbioru, zmienność parametrów oraz wskazać potencjalnie nietypowe obserwacje. Nie pozwalają jeszcze jednoznacznie określić, które zmienne są najważniejsze z punktu widzenia predykcji bankructwa. W kolejnym etapie konieczne będzie porównanie rozkładów poszczególnych parametrów pomiędzy przedsiębiorstwami oznaczonymi jako `alive` i `failed`, a następnie analiza zależności pomiędzy zmiennymi. Dopiero takie zestawienie pozwoli ocenić, które cechy mogą wnosić istotną informację do późniejszego modelu predykcyjnego.